### Lösungen zu Reflection
Schreibe eine Funktion

```python
def reflection(A, B, P):
    '''Spiegle den Punkt P an der Gerade durch A und B und gib ihn zurueck'''
```

<img src="/files/images/reflection.png"><br>

In [3]:
from vector import Vector as Vec
from drawing_helpers import draw_arrow
import widget_helpers as W


def stroke_line(canvas, v, w, color='black', line_width=1):
    canvas.save()
    canvas.stroke_style = color
    canvas.stroke_line(*v, *w)
    canvas.restore()


def reflection(A, B, P):
    '''spiegelt den PUnkt P an der Gerade durch A und B und gibt den PUnkt zurueck'''
    AB = Vec.from_pts(A, B)  # Richtungsvektor AB
    f1 = AB / AB.norm()      # Richtungsvektor mit Laenge 1
    f2 = f1.rot90()

    AP = Vec.from_pts(A, P)  # Richtungsvektor AP
    x, y = AP @ f1, AP @ f2  # Laenge der Projektionen von A auf f1 und f2
    Q = (Vec(*A) + x*f1 - y*f2).as_tuple()
    return Q

In [4]:
A = (20, 30)
B = (90, 60)
P = (70, 30)
O = Vec(*A)

AB = Vec.from_pts(A, B)
AP = Vec.from_pts(A, P)
f1 = AB / AB.norm()
f2 = f1.rot90()


canvas = W.get_canvas()
canvas

Canvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right=…

In [5]:
for pt in (A, B, P):
    canvas.fill_circle(*pt, 2)

canvas.stroke_line(*A, *B)
draw_arrow(canvas, A, P)
stroke_line(canvas, O+30*f2, O-30*f2, color='blue')

In [6]:
Q = reflection(A, B, P)
canvas.fill_circle(*Q, 2)
draw_arrow(canvas, A, Q)

In [7]:
AP_1 = O + (AP @ f1)*f1
AP_2 = O + (AP @ f2)*f2
stroke_line(canvas, P, AP_1, color='lightblue')
stroke_line(canvas, P, AP_2, color='lightblue')

### Lösungen zu Properties (Getters und Setters)

Ein Polygon-Objekt `ploy = Polygon(pts, anchor)` wird erstellt aus einer Menge von Punkten (Ecken) und einem
Ankerpunkt.  
Intern werden alle Punkte als Vektoren in privaten Attributen gespeichert.  Der Benutzer braucht das nicht zu wissen.
Dank Getters und Setters kann z.B. der Ankerpunkt mit `poly.anchor` gelesen und mit `poly.anchor = (1, 2)` geschreiben werden.

Die Klasse `Polygon` hat Getters für folgende Attribute:
- `poly.pts` -> Koordinaten der Ecken als Tuple,
- `poly.anchor` -> Ankerpunktes asl Tuple,
- `poly.size` -> Grösse relativ zum initialen Polygon,
- `poly.angle` -> Drehwinkel relativ zum initialen Polygon.

und Setters für folgende Attribute:

- `poly.anchor = new_anchor` -> verschiebt den Ankerpunkt **und** alle Ecken,
- `poly.size = 2` -> setzt `size` **und** streckt alle Punkte von Ankerpunkt (die initialen Längen werden verdoppelt),
- `poly.angle = math.pi/6` -> setzt `angle` **und** dreht das Polygon um den Ankerpunkt.

Das Polygon hat eine Methode  
- `draw(self, canvas, clear=True,  fill_style=None, stroke_style='black')`,
  
die das Polygon auf die Leinwand malt.

In [16]:
class Polygon:
    def __init__(self, pts, anchor):
        self._vecs = [Vec(*pt) for pt in pts]  # private Attribute
        self._anchor = Vec(*anchor)
        self._size = 1
        self._angle = 0

    def _translate_vecs(self, dv):
        '''dv: Verschiebungsvektor'''
        self._vecs = [v + dv for v in self._vecs]

    def _scale_vecs(self, scale):
        '''scale: Streckungsfaktor
           streckt Punkte von anchor aus
        '''
        c = self._anchor
        self._vecs = [scale*(v-c) + c for v in self._vecs]

    def _rotate_vecs(self, alpha):
        '''rotiere alle Punkte den Ankerpunkt'''
        c = self._anchor
        self._vecs = [(v-c).rotate(alpha) + c for v in self._vecs]

    @property
    def pts(self):
        return [v.as_tuple() for v in self._vecs]

    @property
    def anchor(self):
        return self._anchor.as_tuple()

    @anchor.setter
    def anchor(self, pos):
        new_anchor = Vec(*pos)
        dv = new_anchor - self._anchor  # self._anchor + dv = new_anchor
        self._translate_vecs(dv)
        self._anchor = new_anchor

    @property
    def size(self):
        return self._size

    @size.setter
    def size(self, new_size):
        scale_factor = new_size / self._size  # self._size * scale_factor = new_size
        self._scale_vecs(scale_factor)
        self._size = new_size

    @property
    def angle(self):
        return self._angle

    @angle.setter
    def angle(self, new_angle):
        phi = new_angle - self._angle  # self._angle + phi = new_angle
        self._rotate_vecs(phi)
        self._angle = new_angle

    def draw(self, canvas, clear=True,  fill_style=None, stroke_style='black'):
        canvas.save()
        if clear:
            canvas.clear()
        if fill_style:
            canvas.fill_style = fill_style
            canvas.fill_polygon(self.pts)
        if stroke_style:
            canvas.stroke_style = stroke_style
            canvas.stroke_polygon(self.pts)
        canvas.restore()

    def __repr__(self):
        return f'Polygon(anchor={self.anchor}, pts={self.pts})'

In [17]:
import widget_helpers as W


canvas = W.get_canvas()
canvas

Canvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right=…

In [18]:
pts_F = [(0, 0), (1, 0), (1, 2), (3, 2), (3, 3), (1, 3), (1, 4), (4, 4), (4, 5), (0, 5)]

F = Polygon(pts_F, (.5, 0))
F.anchor = (50, 50)
F.size = 5
F.draw(canvas)

for i, color in enumerate(['red', 'blue']):
    F.angle = (i+1)*2*Vec.PI / 3
    F.draw(canvas, clear=False, fill_style=color)